# 01 – Sentiment-baseline for russiske tekster

Denne notebooken er et **hobbyprosjekt** og brukes kun til **egen læring og utforsking**.  
Målet er å bygge en første, enkel baseline for sentimentanalyse på russiskspråklige tekster.

Jeg ønsker å:

- øve på praktisk NLP med ekte data
- bli tryggere på arbeidsflyten:
  - hente datasett
  - forberede tekstdata
  - trene og evaluere en modell
- ha et konkret utgangspunkt jeg senere kan sammenligne mer avanserte modeller med  
  (f.eks. BERT-baserte modeller for russisk)

Prosjektet er ikke laget for produksjon, men for å:

- forstå styrker og svakheter ved en helt enkel modell
- bygge opp en portefølje av små, gjennomførte analyser

---

## Hva denne notebooken gjør

I grove trekk gjør notebooken følgende:

1. **Laster inn et annotert russisk sentiment-datasett**  
   – via `datasets`-biblioteket (Hugging Face), ("k1tub/sentiment_dataset")

2. **Utforsker datasettet kort**  
   - ser på noen eksempler (tekst + label)  
   - undersøker fordelingen av sentimentklasser (f.eks. positiv, negativ, nøytral)

3. **Forbereder data for modelltrening**  
   - deler i trenings- og testsett (eller bruker ferdige splits)  
   - tilpasser tekst og labels til formatet som `scikit-learn` forventer

4. **Bygger en enkel baseline-modell**  
   - representerer tekst som TF–IDF-vektorer (ord og eventuelt bigrams)  
   - trener en logistisk regresjon som predikerer sentiment basert på disse vektorene

5. **Evaluerer modellen**  
   - måler:
     - nøyaktighet (accuracy)
     - precision, recall og $F_1$-score per klasse og som gjennomsnitt  
   - ser kort på hvilke klasser modellen håndterer bedre/dårligere

6. **Oppsummerer resultatene**  
   - enkel refleksjon over:
     - hvor “bra” en slik baseline egentlig er
     - hva som kan forbedres i senere eksperimenter (f.eks. bedre features eller dypere modeller)

---

## Avgrensninger og intensjon

- Notebooken er **ikke** ment som et ferdig forskningsprosjekt, men som et **læringsverktøy**.
- Koden og analysene er skrevet for å være:
  - forståelige for meg selv i etterkant
  - en del av en offentlig portefølje som viser hvordan jeg jobber
- Datasettet og modellene brukes utelukkende til:
  - å undersøke språklige mønstre og sentiment i russiske tekster
  - å lære mer om praktisk NLP og maskinlæring

Senere vil jeg kunne:

- forbedre denne baselinen
- sammenligne med mer avanserte modeller
- koble resultatene mot større mål i prosjektet (f.eks. analyse av propaganda, mediebias og retorikk om krigen i Ukraina).


In [ ]:
# Grunnleggende pakker
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

import numpy as np
import pandas as pd

import sys
import platform

In [ ]:
# Laster "k1tub/sentiment_dataset" fra Hugging Face
dataset = load_dataset("k1tub/sentiment_dataset")

dataset

In [ ]:
# Første eksempel
example = dataset["train"][0]
example

In [ ]:
# Kolonnenavn
dataset["train"].column_names

In [ ]:
# Unike label-verdier
unique_labels = set(dataset["train"]["label"])
unique_labels

In [ ]:
from sklearn.model_selection import train_test_split

# Henter alle tekster og labels fra train-splitten
texts = dataset["train"]["text"]
labels = dataset["train"]["label"]

# Deler i train og test
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size = 0.2,      # 20 % til test
    random_state = 42,    # gjør delingen reproduserbar
    stratify = labels     # beholder label-fordelingen
)

len(X_train), len(X_test)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Enkel baseline-pipeline: TF–IDF + logistisk regresjon
baseline_model = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=50_000,   # øvre grense på antall features
                ngram_range=(1, 2),    # unigrams + bigrams
                lowercase=True
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                n_jobs=-1,
                class_weight="balanced"  # håndterer evt. ubalanse mellom klasser
            ),
        ),
    ]
)

baseline_model

In [ ]:
%%time

baseline_model.fit(X_train, y_train)

In [20]:
y_pred = baseline_model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.61      0.62      0.62     19318
           1       0.82      0.81      0.81     19375
           2       0.72      0.72      0.72     19399

    accuracy                           0.72     58092
   macro avg       0.72      0.72      0.72     58092
weighted avg       0.72      0.72      0.72     58092

